In [ ]:
using NeuroDSL
include("julius_viz.jl")   # Assurez-vous que julius_viz.jl est dans le même dossier

# -----------------------------------------------------------------------------
# Fonctions de visualisation (si absentes de julius_viz.jl)
# -----------------------------------------------------------------------------
function fix_get_all_nodes(g::NeuroGraph; namespace::Symbol = g.active_ns)
    return Set(keys(g.nodes[namespace]))
end

function fix_get_edges(g::NeuroGraph; namespace::Symbol = g.active_ns)
    edges = Vector{Tuple{Symbol,Symbol}}()
    for rule in values(g.rules[namespace])
        for inp in rule.inputs
            push!(edges, (inp, rule.output))
        end
    end
    return edges
end

function fix_assign_layers(g::NeuroGraph; namespace::Symbol = g.active_ns)
    order = topo_order!(g; namespace = namespace)
    layers = Dict{Symbol, Int}()
    for sym in order
        if haskey(g.rules[namespace], sym)
            max_depth = 0
            for inp in g.rules[namespace][sym].inputs
                max_depth = max(max_depth, get(layers, inp, 0))
            end
            layers[sym] = max_depth + 1
        else
            layers[sym] = 0
        end
    end
    return layers
end

function fix_node_kind(g::NeuroGraph, sym::Symbol; namespace::Symbol = g.active_ns)
    nd = g.nodes[namespace][sym]
    if nd.is_param
        return "param"
    elseif !haskey(g.rules[namespace], sym)
        return "input"
    else
        # Un nœud avec règle est "computed" s'il sert d'entrée à une autre règle
        is_used = any(sym in rule.inputs for rule in values(g.rules[namespace]))
        return is_used ? "computed" : "output"
    end
end

# -----------------------------------------------------------------------------
# Construction d'un petit MLP
# -----------------------------------------------------------------------------
g = NeuroGraph()

# Entrée (batch=32, features=128)
set!(g, :x, randn(Float32, 32, 128))

# Première couche linéaire 128 → 64
linear1 = Linear(128, 64)
h1 = linear1(g, :x, :l1)

# Activation SwiGLU (nécessite deux entrées : gate et up)
# On crée deux copies (via :identity) pour l'exemple
gate = Symbol(:gate)
up   = Symbol(:up)
addrule!(g, GraphRule(gate, [h1], :identity))
addrule!(g, GraphRule(up,   [h1], :identity))
act  = Symbol(:act)
addrule!(g, GraphRule(act, [gate, up], :swiglu))

# Deuxième couche linéaire 64 → 10
linear2 = Linear(64, 10)
out = linear2(g, act, :l2)

# Perte MSE (pour avoir un puits)
set!(g, :target, randn(Float32, 32, 10))
loss = Symbol(:loss)
addrule!(g, GraphRule(loss, [out, :target], :mse_loss))

# -----------------------------------------------------------------------------
# Visualisation
# -----------------------------------------------------------------------------
save_graph_html(g, "mlp_graph.html";
                namespace = :default,
                title     = "MLP avec NeuroDSL",
                open_browser = true)

✅ Sauvegardé : mlp_graph.html
   :default  12 nœuds · 12 arêtes · profondeur 6


"mlp_graph.html"